# 🚀 سامانه فوق سریع Kaggle: ارتقای صوت دروس با هوش مصنوعی (۴ هسته همزمان)
### پایگاه آثار علمی حجت الاسلام دکتر زنجیر زن حسینی

> 💡 **مزیت کگل نسبت به کولب:**
> کگل به طور پیش‌فرض **۴ هسته CPU**، **۳۰ گیگابایت رم** و دیسک بسیار پرسرعت NVMe ارائه می‌دهد.  
> با اجرای همزمان این نوتبوک در کگل برای **بخش دوم (دوره‌های ۲۳ تا ۴۶)** و اجرای نوتبوک کولب برای **بخش اول (دوره‌های ۱ تا ۲۲)**، کل دروس در کمتر از یک ساعت پایان می‌یابند!


### 🔹 مرحله اول: نصب ابزارها و دانلود مدل شبکه عصبی
سلول زیر در چند ثانیه FFmpeg و مدل شبکه عصبی RNNoise را آماده می‌سازد.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://raw.githubusercontent.com/GregorR/rnnoise-models/master/somnolent-hogwash-2018-09-01/sh.rnnn -O /tmp/rnnoise_model.rnnn
print('✅ مدل RNNoise و FFmpeg در کمتر از ۳ ثانیه آماده شدند!')

### 🔹 مرحله دوم: اتصال یا کپی پوشه صوت‌ها
اگر فایل‌ها را از درایو با PyDrive2 یا rclone یا gdown متصل کرده‌اید، مسیر آن را مشخص نمایید.
سلول زیر پکیج کمکی اتصال به گوگل درایو را آماده می‌کند.

In [ ]:
!pip install -q pydrive2
print('✅ ابزار اتصال گوگل درایو نصب شد.')

### 🔹 مرحله سوم: اجرای پردازش ۴ هسته‌ای موازی در کگل
گزینه `1` را برای اجرای خودکار بخش دوم دروس (عرفان، تفسیر، اصول، فقه) انتخاب نمایید.

In [ ]:
# -*- coding: utf-8 -*-
"""
🎙️ سامانه اختصاصی Kaggle: شتاب‌دهی چندرشته‌ای هوش مصنوعی صوت‌های دروس (Kaggle 4-Core Worker Pool)
استاد: حجت الاسلام دکتر زنجیر زن حسینی
محیط اجرا: Kaggle Notebook (دارای ۴ هسته اختصاصی vCPU، ۳۰ گیگابایت رم، و دیسک پرسرعت NVMe)
حساب گوگل درایو: mahdin.dev@gmail.com
"""

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# ۱. راه‌اندازی پوشه‌ها در کگل
KAGGLE_WORK = '/kaggle/working'
TEMP_WORK_DIR = '/tmp/audio_worker'
os.makedirs(TEMP_WORK_DIR, exist_ok=True)

# نحوه دسترسی به فایل‌ها در کگل:
# اگر از Google Drive با PyDrive یا gdown استفاده می‌کنید:
SOURCE_BASE = '/kaggle/working/Dorus_Ostad_Zanjirzan'
TARGET_BASE = '/kaggle/working/Dorus_Ostad_Zanjirzan_Enhanced'

# اگر ورودی به عنوان Dataset به کگل اضافه شده باشد:
if os.path.exists('/kaggle/input/dorus-ostad-zanjirzan'):
    SOURCE_BASE = '/kaggle/input/dorus-ostad-zanjirzan'
    TARGET_BASE = '/kaggle/working/Dorus_Ostad_Zanjirzan_Enhanced'

os.makedirs(TARGET_BASE, exist_ok=True)

print(f"📁 پوشه صوت‌های خام:\n   {SOURCE_BASE}")
print(f"✨ پوشه ذخیره صوت‌های ارتقایافته:\n   {TARGET_BASE}")
print(f"⚡ حافظه موقت محلی NVMe:\n   {TEMP_WORK_DIR}\n")

# ۲. دریافت مدل شبکه عصبی RNNoise
RNNOISE_MODEL = '/tmp/rnnoise_model.rnnn'
if not (os.path.exists(RNNOISE_MODEL) and os.path.getsize(RNNOISE_MODEL) > 1000):
    print("🧠 دریافت مدل شبکه عصبی RNNoise...")
    subprocess.run([
        "wget", "-q",
        "https://raw.githubusercontent.com/GregorR/rnnoise-models/master/somnolent-hogwash-2018-09-01/sh.rnnn",
        "-O", RNNOISE_MODEL
    ])

has_rnnoise = os.path.exists(RNNOISE_MODEL) and os.path.getsize(RNNOISE_MODEL) > 1000

def _worker_task(args):
    src_file, dst_file, worker_id = args
    base_name = os.path.splitext(os.path.basename(src_file))[0]
    
    if os.path.exists(dst_file) and os.path.getsize(dst_file) > 100 * 1024:
        return {"status": "SKIPPED", "file": base_name, "time": 0}
        
    start_t = time.time()
    ext = os.path.splitext(src_file)[1]
    local_in = os.path.join(TEMP_WORK_DIR, f"in_k{worker_id}_{os.getpid()}{ext}")
    local_out = os.path.join(TEMP_WORK_DIR, f"out_k{worker_id}_{os.getpid()}.mp3")
    
    try:
        shutil.copyfile(src_file, local_in)
        if has_rnnoise:
            audio_filter = f"highpass=f=75,arnndn=m={RNNOISE_MODEL}:mix=0.88,dynaudnorm=f=200:g=11:m=10"
        else:
            audio_filter = "highpass=f=75,afftdn=nr=18:nf=-35:tn=enabled,dynaudnorm=f=200:g=11:m=10"
            
        cmd = [
            "ffmpeg", "-y", "-i", local_in,
            "-af", audio_filter,
            "-codec:a", "libmp3lame", "-b:a", "192k", "-ar", "48000",
            local_out
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
        if os.path.exists(local_out) and os.path.getsize(local_out) > 50 * 1024:
            shutil.copyfile(local_out, dst_file)
            elapsed = time.time() - start_t
            size_mb = os.path.getsize(dst_file) / (1024 * 1024)
            return {"status": "SUCCESS", "file": base_name, "time": elapsed, "size_mb": size_mb}
        else:
            return {"status": "FAILED", "file": base_name, "error": "خروجی نامعتبر"}
    except Exception as e:
        return {"status": "FAILED", "file": base_name, "error": str(e)}
    finally:
        for tmp in (local_in, local_out):
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except Exception:
                    pass

def process_course_parallel(cat_name, course_name, max_workers=4):
    src_dir = os.path.join(SOURCE_BASE, cat_name, course_name)
    dst_dir = os.path.join(TARGET_BASE, cat_name, course_name)
    os.makedirs(dst_dir, exist_ok=True)
    
    if not os.path.exists(src_dir):
        print(f"   ⚠️ پوشه یافت نشد: {src_dir}")
        return
        
    files = sorted([f for f in os.listdir(src_dir) if f.lower().endswith(('.mp3', '.wav', '.m4a', '.ogg'))])
    if not files:
        print(f"   ℹ️ هیچ فایل صوتی در این پوشه نیست.")
        return
        
    print(f"\n" + "="*70)
    print(f"🎙️ پردازش ۴ هسته‌ای کگل: {course_name} ({len(files)} جلسه)")
    print(f"   📂 شاخه: {cat_name}/{course_name} | ⚙️ ورکر همزمان: {max_workers}")
    print("="*70)
    
    tasks = []
    for idx, fname in enumerate(files):
        src_file = os.path.join(src_dir, fname)
        base_name = os.path.splitext(fname)[0]
        dst_file = os.path.join(dst_dir, f"{base_name}.mp3")
        tasks.append((src_file, dst_file, idx % max_workers))
        
    c_start = time.time()
    succ = 0
    skip = 0
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_worker_task, t): t for t in tasks}
        for f in as_completed(futures):
            res = f.result()
            if res["status"] == "SKIPPED":
                skip += 1
                print(f"   ⏩ [قبلاً ذخیره شده]: {res['file']}.mp3")
            elif res["status"] == "SUCCESS":
                succ += 1
                print(f"   ✨ [تکمیل شد]: {res['file']}.mp3 ({res['size_mb']:.1f} MB) | زمان: {res['time']:.1f}s")
            else:
                print(f"   ❌ [خطا]: {res['file']} -> {res.get('error')}")
    print(f"🏁 پایان دوره {course_name}: {succ} ارتقایافته، {skip} ردشده | زمان: {time.time()-c_start:.1f} ثانیه")

COURSES_CATALOG = [
    {"category": "01_منطق", "courses": [{"title": "مبادی_منطق"}, {"title": "منطق_مظفر_مادی"}, {"title": "حاشیه_ملاعبدالله"}, {"title": "جوهر_النضید"}, {"title": "منطق_مظفر_صوری"}]},
    {"category": "02_فلسفه", "courses": [{"title": "مبانی_حکمت"}, {"title": "الاشارات_و_التنبیهات_۱۰_نمط"}, {"title": "الشواهد_الربوبیه_سال_اول"}, {"title": "الشواهد_الربوبیه_سال_دوم"}, {"title": "منظومه_حکمت"}, {"title": "بدایه_الحکمه"}, {"title": "نهایه_الحکمه_جلد۱"}, {"title": "نهایه_الحکمه_جلد۲"}, {"title": "فلسفه_غرب"}, {"title": "حکمت_اشراق"}, {"title": "الاسفار_الاربعه_جلد۱_سال_اول"}, {"title": "الاسفار_الاربعه_جلد۱_سال_دوم"}, {"title": "خلاصه_نهایه_الحکمه_جلد۱"}, {"title": "بدایه_الحکمه_با_تفکر_آزاد"}, {"title": "الهیات_شفا_سال_اول"}]},
    {"category": "03_کلام", "courses": [{"title": "تجرید_الاعتقاد_کشف_المراد"}]},
    {"category": "04_عرفان_نظری", "courses": [{"title": "مبانی_عرفان"}, {"title": "تمهید_القواعد"}, {"title": "مقدمه_قیصری_بر_فصوص_الحکم"}, {"title": "فصوص_الحکم_سال_اول"}, {"title": "فصوص_الحکم_سال_دوم"}, {"title": "فصوص_الحکم_سال_سوم"}]},
    {"category": "05_عرفان_عملی", "courses": [{"title": "منازل_السائرین"}, {"title": "اوصاف_الاشراف"}]},
    {"category": "06_تفسیر", "courses": [{"title": "تفسیر_سوره_حمد_صدرالمتألهین"}, {"title": "تفسیر_عرفانی_و_انفسی_سوره_بقره"}]},
    {"category": "07_اصول", "courses": [{"title": "حلقه_ثالثه_بخش_۱"}, {"title": "رسائل_۱_قطع_و_ظن"}, {"title": "رسائل_۲_حجیت_خبر_واحد"}, {"title": "انسداد_رسائل"}, {"title": "رسائل_۳_شبهه_وجوبیه"}, {"title": "رسائل_۴_فیما_یعتبر"}, {"title": "رسائل_۵_شروط_استصحاب"}, {"title": "اقوال_استصحاب_رسائل"}, {"title": "کفایه_الاصول_۱"}, {"title": "کفایه_الاصول_۲"}, {"title": "کفایه_الاصول_۴"}, {"title": "کفایه_الاصول_۳"}]},
    {"category": "08_فقه", "courses": [{"title": "مکاسب_۶_خیار_عیب"}, {"title": "مکاسب_۱_مکاسب_محرمه"}, {"title": "مکاسب_۳_شروط_المتعاقدین"}]}
]

def get_all_courses_flat():
    flat = []
    for cat in COURSES_CATALOG:
        for c in cat["courses"]:
            flat.append((cat["category"], c["title"]))
    return flat

def main():
    workers = 4 # Kaggle has 4 dedicated vCPUs
    all_courses = get_all_courses_flat()
    print("="*70)
    print(f"🚀 سامانه پالایش فوق‌سریع ۴ هسته‌ای Kaggle (تعداد کل دوره‌ها: {len(all_courses)})")
    print("   [1] اجرای بخش دوم دوره‌ها: دوره‌های ۲۳ تا ۴۶ (عرفان، تفسیر، اصول، فقه - پیشنهاد اجرای موازی با کولب)")
    print("   [2] اجرای بخش اول دوره‌ها: دوره‌های ۱ تا ۲۲ (منطق، فلسفه، کلام)")
    print("   [3] اجرای پردازش کلی تمام دوره‌ها")
    print("="*70)
    choice = input("👉 گزینه مورد نظر را انتخاب فرمایید (1/2/3): ").strip() or "1"
    
    if choice == "1":
        half2 = all_courses[22:]
        print(f"\n🚀 شروع پالایش بخش دوم در کگل ({len(half2)} دوره)...")
        for cat_name, course_title in half2:
            process_course_parallel(cat_name, course_title, max_workers=workers)
        print("\n🎉 پالایش بخش دوم در کگل با موفقیت به پایان رسید!")
    elif choice == "2":
        half1 = all_courses[:22]
        print(f"\n🚀 شروع پالایش بخش اول در کگل ({len(half1)} دوره)...")
        for cat_name, course_title in half1:
            process_course_parallel(cat_name, course_title, max_workers=workers)
        print("\n🎉 پالایش بخش اول با موفقیت به پایان رسید!")
    else:
        print(f"\n🚀 شروع پالایش تمام دوره‌ها در کگل...")
        for cat_name, course_title in all_courses:
            process_course_parallel(cat_name, course_title, max_workers=workers)
        print("\n🎉 تمام دوره‌ها با موفقیت به پایان رسیدند!")

if __name__ == "__main__":
    main()
